# Baseline — generate MOT15 detections (.npy)

**Что делает этот ноутбук:** один раз прогоняет TensorFlow-модель и создаёт 4 файла `.npy` для MOT15.

**Перед запуском на Google Drive должна лежать папка `resources/`** (вы уже залили — отлично).

**Как открыть:** [ваш Colab](https://colab.research.google.com/drive/15efuo1aEkNo8l5TrqqeE92OSi0yYUJ9X) → Runtime → Change runtime type → **T4 GPU** → Run all.

> Colab работает на серверах Google (GPU NVIDIA), **не на вашем Mac M4**. M4 тут не важен.

In [ ]:
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"

from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir(DRIVE_RESOURCES), f"Папка не найдена: {DRIVE_RESOURCES}"

# Клонируем код (HTTPS, без SSH)
if not os.path.isdir("/content/DeepSORT_Project_CV"):
    !git clone https://github.com/Valeriia-Reznik-Dev/DeepSORT_Project_CV.git /content/DeepSORT_Project_CV

%cd /content/DeepSORT_Project_CV

# Подключаем ваши данные с Drive
if os.path.islink("resources"):
    os.remove("resources")
elif os.path.isdir("resources"):
    !rm -rf resources
!ln -s "{DRIVE_RESOURCES}" resources

print("resources ->", os.path.realpath("resources"))

In [ ]:
!pip install -q -r requirements-gpu.txt pyyaml

In [ ]:
import os

MOT15_SEQUENCES = [
    "TUD-Campus",
    "TUD-Stadtmitte",
    "KITTI-17",
    "PETS09-S2L1",
]

mot_dir = "resources/detections/MOT15/train"
model_path = "resources/networks/mars-small128.pb"
output_dir = "resources/detections/MOT15_train"

assert os.path.isfile(model_path), f"Missing model: {model_path}"
for seq in MOT15_SEQUENCES:
    seq_dir = os.path.join(mot_dir, seq)
    assert os.path.isdir(seq_dir), f"Missing sequence: {seq_dir}"

print("OK: model and MOT15 train sequences found")

In [ ]:
!python tools/generate_detections.py \
    --model=resources/networks/mars-small128.pb \
    --mot_dir=resources/detections/MOT15/train \
    --output_dir=resources/detections/MOT15_train

In [ ]:
import numpy as np
import os

for seq in MOT15_SEQUENCES:
    path = f"resources/detections/MOT15_train/{seq}.npy"
    arr = np.load(path)
    print(seq, arr.shape)

In [ ]:
# Zip outputs for download to local machine
!zip -r MOT15_train_npy.zip resources/detections/MOT15_train/

from google.colab import files
files.download("MOT15_train_npy.zip")

## После скачивания zip на Mac

```bash
cd final_project
unzip ~/Downloads/MOT15_train_npy.zip
pip install -r requirements-baseline.txt
python scripts/run_baseline.py      # сначала 4 MOT15
python scripts/run_overlays.py
```

### Про MOT16-09 и MOT16-11
В `resources` есть только `.npy`, **видео MOT16 нет** — без кадров трекер и overlay не запустить.
Пока работаем с **4 MOT15**; MOT16 добавим, когда появятся видео (регистрация на [motchallenge.net](https://motchallenge.net/data/MOT16/) или зеркало от преподавателя).